From https://adventuresinmachinelearning.com/convolutional-neural-networks-tutorial-in-pytorch/

# Loading the dataset

PyTorch has an integrated MNIST dataset (in the torchvision package) which we can use via the DataLoader functionality. In this sub-section, I’ll go through how to setup the data loader for the MNIST data set. But first, some preliminary variables need to be defined:

In [45]:
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.utils.data as tud


In [46]:
num_epochs = 5
num_classes = 10
batch_size = 100
learning_rate = 0.001

DATA_PATH = '/home/anurag/gitspace/anuragvvworkspace.github.io/subjects/machine_learning/MNIST_Data'
MODEL_STORE_PATH = '/home/anurag/gitspace/anuragvvworkspace.github.io/subjects/machine_learning/MNIST_Data/model'
print(DATA_PATH)
print(MODEL_STORE_PATH)

/home/anurag/gitspace/anuragvvworkspace.github.io/subjects/machine_learning/MNIST_Data
/home/anurag/gitspace/anuragvvworkspace.github.io/subjects/machine_learning/MNIST_Data/model


Next, we setup a transform to apply to the MNIST data, and also the data set variables:


In [52]:
# transforms to apply to the data
trans = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

# MNIST dataset
train_dataset = torchvision.datasets.MNIST(root=DATA_PATH, train=True, transform=trans, download=True)
test_dataset = torchvision.datasets.MNIST(root=DATA_PATH, train=False, transform=trans)
print(5)

5


In [51]:
train_loader = tud.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = tud.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Creating the model

In [49]:
class ConvNet(torch.nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.drop_out = nn.Dropout()
        self.fc1 = nn.Linear(7 * 7 * 64, 1000)
        self.fc2 = nn.Linear(1000, 10)
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.drop_out(out)
        out = self.fc1(out)
        out = self.fc2(out)
        return out

Ok – so this is where the model definition takes place. The most straight-forward way of creating a neural network structure in PyTorch is by creating a class which inherits from the nn.Module super class within PyTorch. The nn.Module is a very useful PyTorch class which contains all you need to construct your typical deep learning networks. It also has handy functions such as ways to move variables and operations onto a GPU or back to a CPU, apply recursive functions across all the properties in the class (i.e. resetting all the weight variables), creates streamlined interfaces for training and so on. It is worth checking out all the methods available here.

The first step is to create some sequential layer objects within the class _init_ function. First, we create layer 1 (self.layer1) by creating a nn.Sequential object. This method allows us to create sequentially ordered layers in our network and is a handy way of creating a convolution + ReLU + pooling sequence. As can be observed, the first element in the sequential definition is the Conv2d nn.Module method – this method creates a set of convolutional filters. The first argument is the number of input channels – in this case, it is our single channel grayscale MNIST images, so the argument is 1. The second argument to Conv2d is the number of output channels – as shown in the model architecture diagram above, the first convolutional filter layer comprises of 32 channels, so this is the value of our second argument.

The kernel_size argument is the size of the convolutional filter – in this case we want 5 x 5 sized convolutional filters – so the argument is 5. If you wanted filters with different sized shapes in the x and y directions, you’d supply a tuple (x-size, y-size). Finally, we want to specify the padding argument. This takes a little bit more thought. The output size of any dimension from either a convolutional filtering or pooling operation can be calculated by the following equation:


W
o
u
t
=
(
W
i
n
–
F
+
2
P
)
/
S
+
1

Where 
W
i
n
 is the width of the input, F is the filter size, P is the padding and S is the stride. The same formula applies to the height calculation, but seeing as our image and filtering are symmetrical the same formula applies to both. If we wish to keep our input and output dimensions the same, with a filter size of 5 and a stride of 1, it turns out from the above formula that we need a padding of 2. Therefore, the argument for padding in Conv2d is 2.

The next element in the sequence is a simple ReLU activation. The last element that is added in the sequential definition for self.layer1 is the max pooling operation. The first argument is the pooling size, which is 2 x 2 and hence the argument is 2. Second – we want to down-sample our data by reducing the effective image size by a factor of 2. To do this, using the formula above, we set the stride to 2 and the padding to zero. Therefore, the stride argument is equal to 2. The padding argument defaults to 0 if we don’t specify it – so that’s what is done in the code above. From these calculations, we now know that the output from self.layer1 will be 32 channels of 14 x 14 “images”.

Next, the second layer, self.layer2, is defined in the same way as the first layer. The only difference is that the input into the Conv2d function is now 32 channels, with an output of 64 channels. Using the same logic, and given the pooling down-sampling, the output from self.layer2 is 64 channels of 7 x 7 images.

Next, we specify a drop-out layer to avoid over-fitting in the model. Finally, two two fully connected layers are created. The first layer will be of size 7 x 7 x 64 nodes and will connect to the second layer of 1000 nodes. To create a fully connected layer in PyTorch, we use the nn.Linear method. The first argument to this method is the number of nodes in the layer, and the second argument is the number of nodes in the following layer.

With this _init_ definition, the layer definitions have now been created. The next step is to define how the data flows through these layers when performing the forward pass through the network:



It is important to call this function “forward” as this will override the base forward function in nn.Module and allow all the nn.Module functionality to work correctly. As can be observed, it takes an input argument x, which is the data that is to be passed through the model (i.e. a batch of data). We pass this data into the first layer (self.layer1) and return the output as “out”. This output is then fed into the following layer and so on. Note, after self.layer2, we apply a reshaping function to out, which flattens the data dimensions from 7 x 7 x 64 into 3164 x 1. Next, the dropout is applied followed by the two fully connected layers, with the final output being returned from the function.



# Train Model

Before we train the model, we have to first create an instance of our ConvNet class, and define our loss function and optimizer:

In [53]:
model = ConvNet()

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

Training Loop

In [54]:
# Train the model
total_step = len(train_loader)
loss_list = []
acc_list = []
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # Run the forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss_list.append(loss.item())

        # Backprop and perform Adam optimisation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track the accuracy
        total = labels.size(0)
        _, predicted = torch.max(outputs.data, 1)
        correct = (predicted == labels).sum().item()
        acc_list.append(correct / total)

        if (i + 1) % 100 == 0:
            print('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}, Accuracy: {:.2f}%'
                  .format(epoch + 1, num_epochs, i + 1, total_step, loss.item(),
                          (correct / total) * 100))

Epoch [1/5], Step [100/600], Loss: 0.1987, Accuracy: 93.00%
Epoch [1/5], Step [200/600], Loss: 0.1150, Accuracy: 99.00%
Epoch [1/5], Step [300/600], Loss: 0.0654, Accuracy: 95.00%
Epoch [1/5], Step [400/600], Loss: 0.0686, Accuracy: 98.00%
Epoch [1/5], Step [500/600], Loss: 0.1393, Accuracy: 96.00%
Epoch [1/5], Step [600/600], Loss: 0.0456, Accuracy: 97.00%
Epoch [2/5], Step [100/600], Loss: 0.0294, Accuracy: 99.00%
Epoch [2/5], Step [200/600], Loss: 0.2094, Accuracy: 95.00%
Epoch [2/5], Step [300/600], Loss: 0.0509, Accuracy: 99.00%
Epoch [2/5], Step [400/600], Loss: 0.0657, Accuracy: 98.00%
Epoch [2/5], Step [500/600], Loss: 0.0564, Accuracy: 98.00%
Epoch [2/5], Step [600/600], Loss: 0.0213, Accuracy: 99.00%
Epoch [3/5], Step [100/600], Loss: 0.0233, Accuracy: 100.00%
Epoch [3/5], Step [200/600], Loss: 0.0550, Accuracy: 98.00%
Epoch [3/5], Step [300/600], Loss: 0.0356, Accuracy: 97.00%
Epoch [3/5], Step [400/600], Loss: 0.1062, Accuracy: 96.00%
Epoch [3/5], Step [500/600], Loss: 0.01